# 22 · Reranker：召回之后的精排

> 向量检索“快而粗”（Bi-Encoder，各自编码），Reranker“慢而准”（Cross-Encoder，成对编码）。两段式是生产 RAG 的标准架构。

**本文件覆盖知识点**：两段式流程(Retriever→Reranker→LLM) / Cross Encoder / Bi-Encoder / Late Interaction / LLM Reranker / BGE·Jina·Cohere·BGE-v2

```text
Query → Retriever → Top 50 → Reranker → Top 5 → LLM
        (粗召回)      (候选池)   (精排)     (进上下文)
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if _CACHE_FILE.exists():
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    return {}

def _save_cache(cache):
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    np.savez_compressed(_CACHE_FILE, hashes=hs, vectors=vs)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. Bi-Encoder vs Cross-Encoder

| | Bi-Encoder | Cross-Encoder |
|--|-----------|---------------|
| 编码 | query、doc **各自**编码成向量 | query 与每个 doc **拼接后**一起编码 |
| 交互 | 向量点积，无深层交互 | 逐 token 深度交互 |
| 精度 | 粗 | 更精细 |
| 速度 | 快（向量可预计算、建索引） | 慢（每条候选都要过一次模型） |
| 用于 | **召回**（Retriever） | **精排**（Reranker） |

> **为什么 bge-reranker 叫 Cross-Encoder？** 因为它把 (query, doc) 拼成一个输入喂给 Transformer，让两边的 token 互相“看见”，从而判断细粒度相关性——代价是无法预计算 doc 向量，只能在线逐条打分，故只能用于少量候选的精排。

In [ ]:
# 两段式实战：真实召回 → qwen3-rerank 精排
# 注：旧模型名 gte-rerank 在本账号返回 403 AccessDenied，已改用同族的 qwen3-rerank。
from dashscope import TextReRank

RERANK_MODEL = 'qwen3-rerank'

def rerank(query, documents, top_n=3, model=RERANK_MODEL):
    """对候选片段按与 query 的相关度精排；失败时退回原顺序（生产必备的降级）"""
    texts = [d['text'] if isinstance(d, dict) else d for d in documents]
    if not texts:
        return []
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        print('重排失败，退回召回顺序:', getattr(r, 'message', ''))
        return [(t, 0.0) for t in texts[:top_n]]
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

query = '星云客服机器人如何计费？'
cands = hybrid_retrieve(query, k=10)                 # ① 粗召回：真实混合检索取 Top-10 候选池
print('① 召回候选池（%d 条，未精排，只按向量/BM25 顺序）:' % len(cands))
for i, c in enumerate(cands, 1):
    print('   %2d. [%s·%s] %s' % (i, c['source'], c['section'], c['text'][:34].replace('\n', ' ')))

if _HAS_KEY:
    ranked = rerank(query, [c['text'] for c in cands], top_n=3)     # ② 精排
    print('\n② qwen3-rerank 精排后 Top-3（越靠前越扣题）:')
    for text, score in ranked:
        print('   %+.3f  %s' % (score, text[:46].replace('\n', ' ')))
else:
    recorded("""② qwen3-rerank 精排后 Top-3（越靠前越扣题）:
   +0.871  ## 计费口径 - **坐席**：按开通的客服坐席账号数计费，机器人自动应答不占用坐席
   +0.827  本文档说明星云智能客服机器人的套餐价格、计费口径、超量计费规则、退款政策与服务等级协议
   +0.804  ## 计费相关 **Q：怎么收费？** A：按版本订阅，基础版 298 元/月、标准版 99""",
             '录制于 2026-09-12，模型 qwen3-rerank')
print('\n→ 精排只作用于小候选池：它比向量召回准，但每条候选都要过一次模型，慢而贵；'
      '注意候选池里混进来的「故障排查」「API 文档」开头段，正是精排要压下去的对象。')

## 2. 主流 Reranker

| 模型 | 类型 | 备注 |
|------|------|------|
| **BGE-reranker / v2 / v2-m3** | Cross | 中文强，v2-m3 多语言 |
| **Cohere Rerank** | 服务 API | 上手简单、多语 |
| **Jina Reranker** | Cross | 8K 长候选 |
| **qwen3-rerank(百炼)** | Cross 服务 | 本课程使用（原 gte-rerank 已下线/受限） |

另外两类：
- **Late Interaction（如 ColBERT）**：doc 仍可预编码，但保留每 token 向量，在线做 MaxSim——召回级也能享受“交互”红利（第 40 课）；
- **LLM Reranker**：直接让大模型对候选排序/打分，灵活但贵，适合小候选集。

In [ ]:
# 知识点·真调说明：LLM Reranker —— 让大模型亲自当“精排器”，把候选按相关性重新排序并剔除噪声
import json as _json
rerank_query = '星云客服机器人标准版支持私有化部署吗？'
rerank_cands = [
    '标准版默认走公有云 SaaS，如需私有化部署需联系销售单独开通。',   # 编号1：正面命中
    '支持免费试用额度，付费套餐分基础版、标准版、专业版三档。',       # 编号2：沾边
    '竞品天穹客服主打私有化，最低 20 万一年。',                        # 编号3：貌似相关实则跑题
    '部署方式支持公有云、专有云与本地化，详见部署指南第三章。',       # 编号4：正面命中
    '标准版 998 元/月，含 5 个坐席与基础报表。',                       # 编号5：沾边
]
print('① 检索召回（原顺序，未经精排）:')
for i, t in enumerate(rerank_cands, 1):
    print('  [%s] %s' % (i, t))
print()
cand_list = '\n'.join('[%s] %s' % (i + 1, t) for i, t in enumerate(rerank_cands))
out = _llm_live(
    prompt='问题：%s\n\n候选片段：\n%s\n\n请扮演精排重排器，把候选按与问题的相关度从高到低排序。' % (rerank_query, cand_list),
    system='你是 RAG 的 LLM Reranker（精排器）。规则：针对给定问题，把候选片段按相关度从高到低重排；'
           '明显不相关的直接剔除、不要排在后面充数。只输出一个 JSON，禁止任何其它文字：'
           '{"ordered_ids": [按相关度从高到低的候选编号数组], "reason": "一句话解释为何这样排"}。',
    fallback='未配置 Key 的固定样例：\n'
             '{"ordered_ids": [1, 4], "reason": "候选1和4正面回答标准版/私有化部署，其余只是沾边或讲竞品。"}',
    temperature=0.1,
)
if out is None:
    out = '{"ordered_ids": [1, 4], "reason": "候选1和4正面回答标准版/私有化部署，其余只是沾边或讲竞品。"}'
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    seg = out[out.find('{'): out.rfind('}') + 1]
    obj = _json.loads(seg)
    ids = obj['ordered_ids']
    print('② LLM Reranker 精排后（越靠前越相关）:')
    for k, cid in enumerate(ids, 1):
        print('  %d. [%s] %s' % (k, cid, rerank_cands[int(cid) - 1][:28]))
    dropped = [c for c in range(1, len(rerank_cands) + 1) if c not in ids]
    print('  被剔除（判为不相关）:', dropped or '无')
    print('  排序理由:', obj.get('reason', ''))
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明对 LLM Reranker 的输出约束还要再收紧。')
print('→ LLM 能“读懂语义”来精排，但每条候选都要过一次模型、慢而贵，所以只能作用于小候选集；'
      '这就是它和可预计算向量的 Cross-Encoder 精排（上面 qwen3-rerank）的取舍。')

## 3. 工程要点

- **候选池大小**：召回 Top-50~100 → 精排只留 Top-3~5（池太浅会漏正确项）；
- **失败降级**：Reranker 挂了要能退回召回顺序（代码里已示范）；
- **缓存打分**：同一 (query, doc) 对缓存结果，减少重复计算；
- **验收**：对比“召回后直接 Top-K”与“再重排”的答案质量/命中率。

## 小结

- 两段式：**Bi-Encoder 召回 + Cross-Encoder 精排**；
- Reranker 精但慢，只作用于小候选池；
- 进阶有 Late Interaction / LLM Rerank。